# Matérias e autorias - Senado/Brasil

Consulta autorias somente dos senadores que tiveram exercício real na 57ª legislatura.

As autorias são filtradas pelo período de exercício. Se a API trouxer uma data da matéria,
o filtro é diário; se só houver o ano, o filtro usa interseção entre o ano e algum período
de exercício do parlamentar.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import tempfile
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "senado"
LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
TMP = Path(tempfile.mkdtemp(prefix="pi_ii_bronze_senado_v2_"))

LEGIS_BASE = "https://legis.senado.leg.br/dadosabertos"
ADM_BASE = "https://adm.senado.gov.br/adm-dadosabertos"

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "PI-II-Univesp-Bronze-Senado-Brasil/2.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def pasta_uf(uf):
    return ROOT / uf.lower()

def normalizar_nome(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

def parse_data_senado(serie):
    texto = serie.astype(str).str.strip()
    saida = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")

    iso = texto.str.match(r"^\d{4}-\d{2}-\d{2}$")

    if iso.any():
        saida.loc[iso] = pd.to_datetime(
            texto.loc[iso],
            format="%Y-%m-%d",
            errors="coerce",
        )

    outros = ~iso & texto.ne("")

    if outros.any():
        saida.loc[outros] = pd.to_datetime(
            texto.loc[outros],
            errors="coerce",
            dayfirst=True,
        )

    return saida

def lista(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def api_get(url, params=None, timeout=(30, 180)):
    r = session.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def extrair_por_chave(obj, chave):
    alvo = normalizar_nome(chave)
    encontrados = []

    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                if normalizar_nome(k) == alvo:
                    if isinstance(v, list):
                        encontrados.extend(
                            [i for i in v if isinstance(i, dict)]
                        )
                    elif isinstance(v, dict):
                        encontrados.append(v)
                walk(v)
        elif isinstance(x, list):
            for item in x:
                walk(item)

    walk(obj)

    unicos = []
    vistos = set()

    for item in encontrados:
        canon = json.dumps(
            item,
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        )
        if canon not in vistos:
            vistos.add(canon)
            unicos.append(item)

    return unicos

def maior_lista_de_dicts(obj):
    listas = []

    def walk(x):
        if isinstance(x, list):
            if x and all(isinstance(i, dict) for i in x):
                listas.append(x)
            for item in x:
                walk(item)
        elif isinstance(x, dict):
            for v in x.values():
                walk(v)

    walk(obj)

    return max(listas, key=len) if listas else []

def achar_coluna(df, candidatos=None, contem_todos=None):
    candidatos = candidatos or []
    mapa = {normalizar_nome(c): c for c in df.columns}

    for nome in candidatos:
        chave = normalizar_nome(nome)
        if chave in mapa:
            return mapa[chave]

    if contem_todos:
        termos = [normalizar_nome(x) for x in contem_todos]
        for c in df.columns:
            nc = normalizar_nome(c)
            if all(t in nc for t in termos):
                return c

    return None

def detectar_ano(df):
    candidatos = [
        c for c in df.columns
        if normalizar_nome(c) in {
            "anomateria", "materiaano", "anoproposicao", "ano"
        }
        or normalizar_nome(c).endswith("anomateria")
    ]

    for c in candidatos:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            return s.astype("Int64")

    datas = [
        c for c in df.columns
        if "data" in normalizar_nome(c)
    ]

    for c in datas:
        dt = pd.to_datetime(
            df[c],
            errors="coerce",
            dayfirst=True,
        )
        if dt.notna().any():
            return dt.dt.year.astype("Int64")

    return pd.Series(
        [pd.NA] * len(df),
        index=df.index,
        dtype="Int64",
    )

def carregar_exercicios():
    path = ROOT / "_senadores_exercicios_brasil.csv"

    if not path.exists():
        raise FileNotFoundError(
            "Execute primeiro o notebook 00 v2."
        )

    df = pd.read_csv(
        path,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    if df.empty:
        raise RuntimeError("Mapa histórico de exercícios está vazio.")

    for c in ["data_inicio_exercicio", "data_fim_exercicio"]:
        df[c] = pd.to_datetime(
            df[c],
            errors="coerce",
        )

    return df

def salvar_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
    )
    print(f"Salvo: {path} | {len(df):,}")

def salvar_manifesto(nome, payload):
    payload = dict(payload)
    payload["gerado_em_utc"] = utc_now()

    path = ROOT / f"_manifest_{nome}_brasil.json"

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    print("Manifesto:", path)

print("Bronze Senado:", ROOT)


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

exercicios = carregar_exercicios()

codigos = sorted(
    exercicios["codigo_parlamentar"]
    .astype(str)
    .str.strip()
    .unique()
)

print("Senadores históricos consultáveis:", len(codigos))


In [ ]:
def buscar_autorias(codigo):
    url = f"{LEGIS_BASE}/senador/{codigo}/autorias.json"

    try:
        payload = api_get(url)
        registros = extrair_por_chave(payload, "Autoria")

        if not registros:
            registros = maior_lista_de_dicts(payload)

        return codigo, registros, None

    except Exception as exc:
        return codigo, [], f"{type(exc).__name__}: {exc}"


In [ ]:
resultados = []
erros = {}

with ThreadPoolExecutor(max_workers=6) as executor:
    futuros = {
        executor.submit(buscar_autorias, codigo): codigo
        for codigo in codigos
    }

    for i, futuro in enumerate(as_completed(futuros), start=1):
        codigo, registros, erro = futuro.result()

        if erro:
            erros[codigo] = erro
        else:
            for row in registros:
                flat = pd.json_normalize([row], sep="_")
                flat["_codigo_parlamentar_consulta"] = codigo
                resultados.append(flat)

        if i % 20 == 0 or i == len(futuros):
            print(f"Senadores processados: {i}/{len(futuros)}")

if resultados:
    autorias = pd.concat(
        resultados,
        ignore_index=True,
        sort=False,
    )
else:
    autorias = pd.DataFrame()

print("Registros de autoria antes do filtro:", len(autorias))
print("Erros:", len(erros))


In [ ]:
if autorias.empty:
    raise RuntimeError(
        "Nenhuma autoria foi coletada."
    )

autorias["_ano_recorte"] = detectar_ano(autorias)

# tenta usar a data de apresentação quando disponível
data_col = achar_coluna(
    autorias,
    [
        "Materia_DataApresentacao",
        "DataApresentacao",
        "Materia_Data",
        "Data",
    ],
    contem_todos=["data"],
)

if data_col:
    datas_autoria = parse_data_senado(
        autorias[data_col]
    )
else:
    datas_autoria = pd.Series(
        pd.NaT,
        index=autorias.index,
    )

# períodos por senador
periodos = {}

for codigo, grupo in exercicios.groupby("codigo_parlamentar"):
    periodos[str(codigo)] = [
        (
            row["data_inicio_exercicio"],
            row["data_fim_exercicio"],
            row["uf"],
        )
        for _, row in grupo.iterrows()
    ]

def ativo_no_registro(codigo, ano, data):
    intervalos = periodos.get(str(codigo), [])

    if not intervalos or pd.isna(ano):
        return False, "", ""

    # se a matéria tem data, usa o dia
    if not pd.isna(data):
        for inicio, fim, uf in intervalos:
            fim_cmp = (
                pd.Timestamp("2027-01-31")
                if pd.isna(fim)
                else fim
            )

            if inicio <= data <= fim_cmp:
                return True, uf, "data"

        return False, "", "data"

    # fallback: pelo menos um dia de exercício no ano
    ano_inicio = pd.Timestamp(f"{int(ano)}-01-01")
    ano_fim = pd.Timestamp(f"{int(ano)}-12-31")

    for inicio, fim, uf in intervalos:
        fim_cmp = (
            pd.Timestamp("2027-01-31")
            if pd.isna(fim)
            else fim
        )

        if inicio <= ano_fim and fim_cmp >= ano_inicio:
            return True, uf, "ano"

    return False, "", "ano"

validos = []
ufs = []
criterios = []

for idx, row in autorias.iterrows():
    codigo = str(row["_codigo_parlamentar_consulta"]).strip()
    ano = row["_ano_recorte"]
    data = datas_autoria.loc[idx]

    ok, uf, criterio = ativo_no_registro(
        codigo,
        ano,
        data,
    )

    validos.append(ok)
    ufs.append(uf)
    criterios.append(criterio)

autorias["_uf_recorte"] = ufs
autorias["_criterio_exercicio"] = criterios
autorias["_registro_valido_leg57"] = validos

fora_exercicio = autorias[
    ~autorias["_registro_valido_leg57"]
    & autorias["_ano_recorte"].isin(ANOS)
].copy()

salvar_csv(
    fora_exercicio,
    ROOT / "_materias_autorias_fora_exercicio.csv",
)

autorias = autorias[
    autorias["_registro_valido_leg57"]
    & autorias["_ano_recorte"].isin(ANOS)
].copy()

cod_materia_col = achar_coluna(
    autorias,
    [
        "Materia_Codigo",
        "CodigoMateria",
        "IdentificacaoMateria_CodigoMateria",
        "Codigo",
    ],
    contem_todos=["codigo", "materia"],
)

resumo = []

for uf in UFS:
    base_uf = autorias[
        autorias["_uf_recorte"].eq(uf)
    ].copy()

    for ano in ANOS:
        recorte = base_uf[
            base_uf["_ano_recorte"].eq(ano)
        ].copy()

        salvar_csv(
            recorte,
            pasta_uf(uf) / f"materias_autorias_{ano}.csv",
        )

        if cod_materia_col and cod_materia_col in recorte.columns:
            materias = recorte.drop_duplicates(
                subset=[cod_materia_col]
            ).copy()
        else:
            materias = recorte.drop_duplicates().copy()

        salvar_csv(
            materias,
            pasta_uf(uf) / f"materias_{ano}.csv",
        )

        resumo.append({
            "uf": uf,
            "ano": ano,
            "autorias": len(recorte),
            "materias": len(materias),
        })

auditoria = {
    "senadores_historicos_consultados": len(codigos),
    "erros": erros,
    "coluna_codigo_materia": cod_materia_col,
    "coluna_data_usada": data_col,
    "datas_validas": int(datas_autoria.notna().sum()),
    "datas_invalidas_ou_vazias": int(datas_autoria.isna().sum()),
    "registros_antes_filtro": len(validos),
    "registros_validos_leg57": len(autorias),
    "registros_2023_2026_fora_exercicio": len(fora_exercicio),
}

with (ROOT / "_auditoria_materias.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        auditoria,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

salvar_manifesto(
    "materias",
    {
        "resumo": resumo,
        "erros": erros,
        "auditoria": auditoria,
    },
)

display(pd.DataFrame(resumo))
